# 01 — Test Pitch Detection

Load a recorded WAV file, run pYIN pitch detection, and plot the pitch contour.
Use this notebook to verify that the detector is tracking the bassoon correctly
before running a full analysis session.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf

from audio.pitch_detector import detect_pitch, hz_to_cents, get_note_name
from visualization.plotter import plot_pitch_contour
from utils.config import SAMPLE_RATE

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load a recording

Point `RECORDING_PATH` at any WAV file in `data/recordings/`.  
If you don't have one yet, the cell below can generate a synthetic sine-wave test tone.

In [ ]:
RECORDING_PATH = None   # e.g. '../data/recordings/my_note.wav'

if RECORDING_PATH is not None:
    audio, sr = sf.read(RECORDING_PATH, dtype='float32')
    if audio.ndim > 1:
        audio = audio[:, 0]   # take first channel if stereo
    print(f'Loaded {RECORDING_PATH}  |  {len(audio)/sr:.2f}s  |  {sr} Hz')
else:
    # Synthetic B♭3 (233.08 Hz) with slight vibrato for testing
    sr = SAMPLE_RATE
    duration = 4.0
    t = np.linspace(0, duration, int(duration * sr), endpoint=False)
    vibrato = 5.0 * np.sin(2 * np.pi * 5.5 * t)          # ±5 Hz at 5.5 Hz rate
    freq_t  = 233.08 + vibrato
    phase   = 2 * np.pi * np.cumsum(freq_t) / sr
    audio   = 0.5 * np.sin(phase).astype(np.float32)
    print(f'Using synthetic B♭3 tone  |  {duration}s  |  {sr} Hz')

## 2. Run pitch detection

In [ ]:
times, frequencies, confidence = detect_pitch(audio, sr=sr)

voiced_mask = ~np.isnan(frequencies)
voiced_pct  = voiced_mask.mean() * 100

print(f'Frames analysed : {len(times)}')
print(f'Voiced frames   : {voiced_mask.sum()}  ({voiced_pct:.1f}%)')
print(f'Mean confidence : {confidence.mean():.3f}')

if voiced_mask.any():
    mean_hz   = float(np.nanmean(frequencies))
    mean_note = get_note_name(mean_hz)
    mean_cents = hz_to_cents(mean_hz)
    print(f'Mean pitch      : {mean_hz:.2f} Hz  →  {mean_note}  ({mean_cents:+.1f} cents from A4)')

## 3. Plot pitch contour

In [ ]:
fig = plot_pitch_contour(times, frequencies, title='Pitch Contour — Test Recording', save=False)
plt.show()

## 4. Confidence over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.fill_between(times, confidence, alpha=0.6, color='steelblue')
ax.axhline(0.1, color='red', linewidth=0.8, linestyle='--', label='Confidence threshold (0.1)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voiced probability')
ax.set_ylim(0, 1.05)
ax.set_title('pYIN Confidence')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Per-frame note names (first 20 voiced frames)

In [ ]:
voiced_times = times[voiced_mask]
voiced_freqs = frequencies[voiced_mask]

print(f'{'Time (s)':>10}  {'Freq (Hz)':>10}  {'Note':>6}  {'Cents from A4':>14}')
print('-' * 48)
for t, f in zip(voiced_times[:20], voiced_freqs[:20]):
    note   = get_note_name(f)
    cents  = hz_to_cents(f)
    print(f'{t:10.3f}  {f:10.2f}  {note:>6}  {cents:+14.1f}')